# veckit — Virtual Embryo Challenge local scorer (tutorial)

`veckit` scores a submitted `.h5ad` against the T1/T2/T3 metric panels **entirely offline**, against
reference files *you* supply — no access to the official (held-out) validation/test data, ever. It's a
real, standalone PyPI package: [pypi.org/project/veckit](https://pypi.org/project/veckit/).

**Three files, three roles**
| flag | role |
|---|---|
| `--input` | the prediction you're scoring |
| `--target` | the pseudo target — what `--input` should have predicted |
| `--reference` (T1/T2) / `--wt` (T3) | the reference expression `de_score`/`de_direction`/`severity_slope` are measured *from* — these are PRIMARY metrics ("did you predict the right **change**", not just "does this look plausible"), and a change can't be computed from a single snapshot |

`--reference`/`--wt` is optional and defaults to `--input` itself if omitted — correct **only** when
you're deliberately testing a no-change baseline (`copy_last`/`wt_identity`), where it exactly reproduces
the official floor numbers. For a real model's prediction, always pass a genuine reference file, or
`de_score`/`de_direction` will misleadingly read as "no predicted change" (exactly 0) regardless of what
your model actually did.

**This is not a preview of your real competition score.** Whatever you pass as `--target` is, by
definition, data you already had, so it's typically an easier question than the real held-out target.

## Install

In [ ]:
!pip install -q veckit

## Get the data

**For a real local score**, register and download the released T1/T2/T3 stages from the official
challenge site — [virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data) (see also
[aristoteleo/virtualembryo](https://github.com/aristoteleo/virtualembryo)) — then point `--input`/
`--target`/`--reference`/`--wt` at your own downloaded files.

**To just try the tool first**, this repo ships a few tiny (150-cell) samples — enough to exercise the
CLI, not enough to mean anything scientifically:

In [ ]:
!mkdir -p sample_data
!wget -q -O sample_data/T1_8.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/sample_8.5.h5ad
!wget -q -O sample_data/T1_9.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/sample_9.5.h5ad
!wget -q -O sample_data/T2_9.25.h5ad https://raw.githubusercontent.com/aristoteleo/veckit/main/sample_heart_9.25.h5ad
!wget -q -O sample_data/T2_9.5.h5ad  https://raw.githubusercontent.com/aristoteleo/veckit/main/sample_heart_9.5.h5ad
!wget -q -O sample_data/T3_wt.h5ad   https://raw.githubusercontent.com/aristoteleo/veckit/main/sample_wt.h5ad
!wget -q -O sample_data/T3_ko.h5ad   https://raw.githubusercontent.com/aristoteleo/veckit/main/sample_mab21l2_ko.h5ad
!ls -la sample_data/

## Task 1 — temporal gene-expression prediction

`--reference` is the genuine preceding stage (E8.5), so `de_score`/`de_direction` are computed
meaningfully, not defaulted. `--input` here is E9.5 itself, standing in for a hypothetically *perfect*
prediction — replace it with your own model's output file.

In [ ]:
!veckit --task T1 \
  --input sample_data/T1_9.5.h5ad \
  --target sample_data/T1_9.5.h5ad \
  --reference sample_data/T1_8.5.h5ad

**Quick check, no explicit `--reference`.** Useful for a fast copy_last-style sanity check on the pipeline itself; `--input` doubles as the reference, so `de_score`/`de_direction` correctly (not misleadingly) read as 0 here:

In [ ]:
!veckit --task T1 --input sample_data/T1_8.5.h5ad --target sample_data/T1_9.5.h5ad

## Task 2 — spatial-temporal multiscale prediction

Same three roles, plus `obsm['spatial_3D']` on every file (already present in the sample data) and
`--setting` (`heart`/`embryo`) to label which scope you're scoring.

In [ ]:
!veckit --task T2 --setting heart \
  --input sample_data/T2_9.5.h5ad \
  --target sample_data/T2_9.5.h5ad \
  --reference sample_data/T2_9.25.h5ad

## Task 3 — mutant perturbation prediction

`--wt` plays `--reference`'s role here: the matched wild type the knockout effect (`de_score`/
`de_direction`/`severity_slope`) is measured against.

In [ ]:
!veckit --task T3 \
  --input sample_data/T3_ko.h5ad \
  --target sample_data/T3_ko.h5ad \
  --wt sample_data/T3_wt.h5ad

## Python API

Same thing, without shelling out — useful inside a training/eval loop. `from veckit import score` mirrors
every CLI flag 1:1 as a keyword argument.

In [ ]:
from veckit import score

result = score(
    task="T1",
    input="sample_data/T1_8.5.h5ad",
    target="sample_data/T1_9.5.h5ad",
    reference="sample_data/T1_8.5.h5ad",
)
result["metrics"]

## Next steps

- Swap `--input` for your own model's prediction (or `input=` in Python).
- Download the real released stages from
  [virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data) and swap `--target`/
  `--reference`/`--wt` for them.
- `veckit --help` / `help(score)` for every flag.
- Full task definitions and metric rationale: [virtualembryo.ai](https://virtualembryo.ai).